# 1x1-conv-channel-reshape — worked example 2: 1x1 conv channel expansion via einsum

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `1x1-conv-channel-reshape`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

A 1x1 conv with `C_out > C_in` projects each pixel's channel vector UP into a larger space (the ResNet bottleneck expand). Because the kernel is 1x1, the operation is purely a per-pixel linear map `out_c = sum_k W[c,k] * x_k` and can be written directly as an einsum over the channel axis, leaving `H` and `W` alone.

## Worked solution

Here we expand `C_in=3` channels up to `C_out=8` and express the whole thing as a single einsum, never flattening the spatial grid.

1. **Collapse the kernel dims of the weight.** `conv.weight` is `(C_out, C_in, 1, 1) = (8, 3, 1, 1)`. Use `rearrange(W, 'o i 1 1 -> o i')` to get the `(8, 3)` projection matrix.

2. **Contract over the input-channel axis with einsum.** Write `einops.einsum(x, Wlin, 'b i h w, o i -> b o h w')`. The repeated index `i` (input channels) is summed; `b`, `h`, `w` are carried through unchanged; the new output-channel axis `o` appears. This is exactly `out[b,o,h,w] = sum_i x[b,i,h,w] * W[o,i]`.

3. **Add the bias along the channel axis.** `conv.bias` has shape `(C_out,)`. To broadcast it against `(B, C_out, H, W)` we reshape it to `(1, C_out, 1, 1)` so it lands on the channel axis only.

4. **Why no spatial term?** A 1x1 kernel has a single tap, so there is no neighbor to sum over — the spatial axes are pure spectators. That is why channel expansion is free of any sliding-window logic.

The output is `(2, 8, 4, 4)` and matches `conv(x)`.

In [ ]:
import torch.nn as nn
import einops

def one_by_one_channel_expand(x, conv):
    Wlin = rearrange(conv.weight, 'o i 1 1 -> o i')
    out = einops.einsum(x, Wlin, 'b i h w, o i -> b o h w')
    if conv.bias is not None:
        out = out + conv.bias.view(1, -1, 1, 1)
    return out

t.manual_seed(0)
conv = nn.Conv2d(3, 8, kernel_size=1)
x = t.randn(2, 3, 4, 4)
out = one_by_one_channel_expand(x, conv)
print('output shape:', tuple(out.shape))
print('matches conv:', t.allclose(out, conv(x), atol=1e-5))